# SCC Signed-Error Grid Viewer

Inspect the signed-error SCC grid. The current SCC threshold is `abs(mean(validation_error) + k * std(validation_error))`, with `min_consecutive = 6` fixed.


In [1]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, clear_output

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPERIMENTS_DIR = PROJECT_ROOT / "results" / "kelmarsh" / "experiments"
SCC_C = 6
DEFAULT_K_VALUES = ",".join(f"{i / 2:g}" for i in range(21))

def list_experiments():
    if not EXPERIMENTS_DIR.exists():
        return []
    return sorted(p.name for p in EXPERIMENTS_DIR.iterdir() if (p / "metadata.json").exists())

def format_number(value):
    return f"{float(value):g}".replace(".", "p")

def grid_label(k_values):
    values = [float(v.strip()) for v in k_values.split(",") if v.strip()]
    return f"signed_meanstd_abslimit_c{SCC_C}_k{format_number(min(values))}-{format_number(max(values))}_step0p5"

def grid_path(run_id, k_values):
    return EXPERIMENTS_DIR / run_id / f"scc_grid_{grid_label(k_values)}.csv"

def run_grid(run_id, k_values, force=False):
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "experiments" / "main_grid_scc_signed_meanstd.py"),
        "--run-id", run_id,
        "--k-values", k_values,
    ]
    if force:
        cmd.append("--force")
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("SCC grid run failed")

def load_grid(run_id, k_values):
    path = grid_path(run_id, k_values)
    if not path.exists():
        raise FileNotFoundError(f"Grid summary not found: {path}")
    return pd.read_csv(path)

def show_grid(run_id, k_values):
    data = load_grid(run_id, k_values).sort_values("k")
    display(Markdown("### Best Settings by F1 Score"))
    display(data.sort_values(["f1_score", "operational_false_alarm_rate"], ascending=[False, True])[[
        "k", "precision", "recall", "f1_score", "detected_events", "total_events",
        "operational_false_alarm_rate", "total_alarm_episodes", "true_alarm_episodes",
        "operational_false_alarm_episodes", "threshold_mean", "threshold_min", "threshold_max",
    ]].head(10).style.format({
        "precision": "{:.2%}",
        "recall": "{:.2%}",
        "f1_score": "{:.2%}",
        "operational_false_alarm_rate": "{:.2%}",
        "threshold_mean": "{:.4f}",
        "threshold_min": "{:.4f}",
        "threshold_max": "{:.4f}",
    }))

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, constrained_layout=True)
    axes[0].plot(data["k"], data["f1_score"], marker="o", label="F1 score", color="#1f77b4")
    axes[0].plot(data["k"], data["precision"], marker="o", label="Precision", color="#2ca02c")
    axes[0].plot(data["k"], data["recall"], marker="o", label="Recall", color="#ff7f0e")
    axes[0].set_ylabel("Score")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    axes[0].set_title("SCC performance vs k")

    axes[1].plot(data["k"], data["operational_false_alarm_rate"], marker="o", label="Operational FA rate", color="#d62728")
    axes[1].plot(data["k"], data["miss_rate"], marker="o", label="Miss rate", color="#9467bd")
    axes[1].set_xlabel("k")
    axes[1].set_ylabel("Rate")
    axes[1].set_ylim(0, 1.05)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    plt.show()

def load_validation_predictions(run_id):
    metadata = pd.read_json(EXPERIMENTS_DIR / run_id / "metadata.json", typ="series")
    paths = metadata.get("paths", {})
    rel_path = paths.get("predictions") if isinstance(paths, dict) else None
    path = PROJECT_ROOT / rel_path if rel_path else EXPERIMENTS_DIR / run_id / "healthy_val_predictions.csv"
    return pd.read_csv(path)

def target_from_error_col(col):
    return col.split("::", 1)[1] if "::" in col else col

def show_validation_error_distribution(run_id):
    pred = load_validation_predictions(run_id)
    error_cols = [col for col in pred.columns if col.startswith("error::")]
    if not error_cols:
        raise ValueError("No error:: columns found in validation predictions.")
    fig, axes = plt.subplots(len(error_cols), 1, figsize=(12, 3.2 * len(error_cols)), constrained_layout=True)
    if len(error_cols) == 1:
        axes = [axes]
    rows = []
    for ax, col in zip(axes, error_cols):
        values = pd.to_numeric(pred[col], errors="coerce").dropna()
        target = target_from_error_col(col)
        rows.append({
            "target": target,
            "samples": len(values),
            "mean": values.mean(),
            "std": values.std(ddof=0),
            "median": values.median(),
            "skewness": values.skew(),
        })
        ax.hist(values, bins=120, color="#4c78a8", alpha=0.75)
        ax.axvline(0, color="black", linewidth=1, linestyle="--", label="zero")
        ax.axvline(values.mean(), color="#d62728", linewidth=1.5, label="mean")
        ax.set_title(target)
        ax.set_xlabel("validation_error = y_true_val - y_pred_val")
        ax.set_ylabel("count")
        ax.legend()
    display(pd.DataFrame(rows).style.format({
        "mean": "{:.4f}", "std": "{:.4f}", "median": "{:.4f}", "skewness": "{:.3f}"
    }))
    plt.show()


In [2]:
runs = list_experiments()
if not runs:
    raise FileNotFoundError(f"No experiment metadata found in {EXPERIMENTS_DIR}")

if HAS_WIDGETS:
    run_dropdown = widgets.Dropdown(options=runs, value=runs[-1], description="Run")
    k_text = widgets.Text(value=DEFAULT_K_VALUES, description="k values", layout=widgets.Layout(width="680px"))
    force_checkbox = widgets.Checkbox(value=False, description="force recompute")
    run_grid_button = widgets.Button(description="Run/refresh SCC grid", button_style="primary")
    load_grid_button = widgets.Button(description="Load F1 plot")
    show_dist_button = widgets.Button(description="Show validation error distribution")
    output = widgets.Output()

    def on_run_grid(_):
        with output:
            clear_output(wait=True)
            run_grid(run_dropdown.value, k_text.value, force=force_checkbox.value)
            show_grid(run_dropdown.value, k_text.value)

    def on_load_grid(_):
        with output:
            clear_output(wait=True)
            show_grid(run_dropdown.value, k_text.value)

    def on_show_dist(_):
        with output:
            clear_output(wait=True)
            show_validation_error_distribution(run_dropdown.value)

    run_grid_button.on_click(on_run_grid)
    load_grid_button.on_click(on_load_grid)
    show_dist_button.on_click(on_show_dist)
    display(widgets.VBox([
        widgets.HBox([run_dropdown]),
        widgets.HBox([k_text]),
        widgets.HBox([run_grid_button, load_grid_button, show_dist_button, force_checkbox]),
        output,
    ]))
else:
    print("ipywidgets is not available. Use run_grid(...), show_grid(...), and show_validation_error_distribution(...) manually.")
